In [1]:
import os

os.chdir('..')
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # set for determinism

## Loading the Dataset:

In this section the pointcloud is loaded. The SIREN paper suggests normalizing the point coordinates as periodic activations implicitly expect a bounded input. 

In [2]:
import open3d as o3d
import numpy as np
import torch
import matplotlib.pyplot as plt
import src.model.SIREN as si
from src.model.training import train
import src.loss.SDF_loss as loss
from src.mesh_extraction.marching_cubes_test import write_obj
import src.model.MLP as simple
import src.data.dataset as data
import src.model.pruning_module as pm
import src.mesh_extraction.marching_cubes_gpu as marching_cubes
import random

def set_seeds():
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

mesh = data.MeshDataset('data/pointclouds/Armadillo/Armadillo.ply')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Defining the Model

In this cell we will define the SIREN model. This particular INR uses sine activations for nonlinearity and is supposed to capture more information given the underlying data when compared to a model that uses ReLU activations. This way, a good INR accuracy can be achieved with fewer neurons.

In [3]:
size_per_layer = 256
set_seeds()
model_no = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=1, lambda_normal=2, lambda_inter=0.4, lambda_off=7, lambda_twd=1e-4, model=model_no)
optimizer = torch.optim.Adam(model_no.parameters(), lr=1e-4)
si.weight_stats(model_no)

Weight statistics per layer:
------------------------------------------------------------
Hidden layer  0
  Weight: mean=7.1341e-03, std=1.8916e-01, min=-3.3078e-01, max=3.3230e-01
  Bias  : mean=-1.2152e-02, std=1.8760e-01, min=-3.2975e-01, max=3.3306e-01
  Omega scale (exp): mean=2.7183e+00, std=0.0000e+00
------------------------------------------------------------
Hidden layer  1
  Weight: mean=6.2948e-06, std=2.9461e-03, min=-5.1030e-03, max=5.1029e-03
  Bias  : mean=3.2051e-05, std=2.9077e-03, min=-5.0769e-03, max=5.0955e-03
------------------------------------------------------------
Hidden layer  2
  Weight: mean=-2.4295e-05, std=2.9552e-03, min=-5.1031e-03, max=5.1029e-03
  Bias  : mean=-4.6270e-05, std=2.9337e-03, min=-5.0569e-03, max=5.1006e-03
------------------------------------------------------------
Hidden layer  3
  Weight: mean=-1.7248e-05, std=2.9479e-03, min=-5.1030e-03, max=5.1028e-03
  Bias  : mean=1.3535e-04, std=2.7929e-03, min=-4.9978e-03, max=5.0838e-03
------

/home/nikola/adaptive_pruning_SIREN/src/model/SIREN.py:117: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  print(f"  Bias  : mean={b.mean():.4e}, std={b.std():.4e}, min={b.min():.4e}, max={b.max():.4e}")


## Model training without pruning or densification



In [4]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_no, loss=model_loss, optimizer=optimizer, scene=mesh.scene)

Step 0 | Loss 3.3776051998138428
Step 10 | Loss 0.574488639831543
Step 20 | Loss 0.36560797691345215
Step 30 | Loss 0.2644067406654358
Step 40 | Loss 0.20597997307777405
Step 50 | Loss 0.17927072942256927
Step 60 | Loss 0.1613643765449524
Step 70 | Loss 0.1464022994041443
Step 80 | Loss 0.1297806203365326
Step 90 | Loss 0.12005391716957092
Step 100 | Loss 0.10722192376852036
Step 110 | Loss 0.09953023493289948
Step 120 | Loss 0.09126383811235428
Step 130 | Loss 0.08362793177366257
Step 140 | Loss 0.07998579740524292
Step 150 | Loss 0.07421259582042694
Step 160 | Loss 0.07103084027767181
Step 170 | Loss 0.06612478196620941
Step 180 | Loss 0.06311621516942978
Step 190 | Loss 0.06049710884690285
Step 200 | Loss 0.05794624984264374
Step 210 | Loss 0.05601947754621506
Step 220 | Loss 0.054776497185230255
Step 230 | Loss 0.05283135175704956
Step 240 | Loss 0.050813622772693634
Step 250 | Loss 0.048961322754621506
Step 260 | Loss 0.048248786479234695
Step 270 | Loss 0.047146186232566833
Step 

#### Model size after pruning

In [5]:
model_no.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  256 neurons
Hidden layer  2:  256 neurons
Hidden layer  3:  256 neurons
Hidden layer  4:  256 neurons
Final layer    :    1 neurons


In [5]:
marching_cubes.write_obj("armadillo_128_unpruned.obj", model=model_no, resolution=128, level=0.0)

## Model training with densification

In [4]:
set_seeds()
model_yes = si.SIRENSDF()
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_yes)
optimizer = torch.optim.Adam(model_yes.parameters(), lr=1e-4)
si.weight_stats(model_yes)

Weight statistics per layer:
------------------------------------------------------------
Hidden layer  0
  Weight: mean=7.1341e-03, std=1.8916e-01, min=-3.3078e-01, max=3.3230e-01
  Bias  : mean=-1.2152e-02, std=1.8760e-01, min=-3.2975e-01, max=3.3306e-01
  Omega scale (exp): mean=2.7183e+00, std=0.0000e+00
------------------------------------------------------------
Hidden layer  1
  Weight: mean=6.2948e-06, std=2.9461e-03, min=-5.1030e-03, max=5.1029e-03
  Bias  : mean=3.2051e-05, std=2.9077e-03, min=-5.0769e-03, max=5.0955e-03
------------------------------------------------------------
Hidden layer  2
  Weight: mean=-2.4295e-05, std=2.9552e-03, min=-5.1031e-03, max=5.1029e-03
  Bias  : mean=-4.6270e-05, std=2.9337e-03, min=-5.0569e-03, max=5.1006e-03
------------------------------------------------------------
Hidden layer  3
  Weight: mean=-1.7248e-05, std=2.9479e-03, min=-5.1030e-03, max=5.1028e-03
  Bias  : mean=1.3535e-04, std=2.7929e-03, min=-4.9978e-03, max=5.0838e-03
------

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_yes, loss=model_loss, optimizer=optimizer, scene=mesh.scene, densification=True)

Step 0 | Loss 2.5587291717529297
Step 10 | Loss 0.4769289195537567
Step 20 | Loss 0.37794047594070435
Step 30 | Loss 0.33678868412971497
Step 40 | Loss 0.3176746368408203
Step 50 | Loss 0.3009178042411804
Step 60 | Loss 0.2947434186935425
Step 70 | Loss 0.2815569341182709
Step 80 | Loss 0.2748466432094574
Step 90 | Loss 0.2697475552558899
Step 100 | Loss 0.26551488041877747
Step 110 | Loss 0.2625808119773865
Step 120 | Loss 0.25534701347351074
Step 130 | Loss 0.2555489242076874
Step 140 | Loss 0.25084608793258667
Step 150 | Loss 0.25337663292884827
Step 160 | Loss 0.24703876674175262
Step 170 | Loss 0.2359398603439331
Step 180 | Loss 0.24226884543895721
Step 190 | Loss 0.23784910142421722
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.23929759860038757
Step 210 | Loss 0.295942097902298
Step 220 | Loss 0.2705099284648895
Step 230 | Loss 0.2554838955402374
Step 240 | Loss 0.24585115909576416
Step 250 | Loss 0.23829390108585358
Step 260 | Loss 0.23150791227817535
Step 270 

In [6]:
model_yes.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  256 neurons
Hidden layer  2:  256 neurons
Hidden layer  3:  256 neurons
Hidden layer  4:  256 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_densified.obj", model=model_yes, resolution=128, level=0.0)

## AIRe: Model training (no densification)

In [4]:
set_seeds()
model_aire_no = si.SIRENSDF()
prune_AIRe = pm.AIRe(model_aire_no, 0.6, int(size_per_layer * 0.2))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_aire_no, pruning_module=prune_AIRe)
optimizer = torch.optim.Adam(model_aire_no.parameters(), lr=1e-4)

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_aire_no, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_AIRe)

Step 0 | Loss 2.571408987045288
Step 10 | Loss 0.48973679542541504
Step 20 | Loss 0.39074957370758057
Step 30 | Loss 0.34962552785873413
Step 40 | Loss 0.330488920211792
Step 50 | Loss 0.31372010707855225
Step 60 | Loss 0.3075138330459595
Step 70 | Loss 0.2943316400051117
Step 80 | Loss 0.28757527470588684
Step 90 | Loss 0.28246963024139404
Step 100 | Loss 0.27817487716674805
Step 110 | Loss 0.2752159535884857
Step 120 | Loss 0.26794353127479553
Step 130 | Loss 0.2681148052215576
Step 140 | Loss 0.26337018609046936
Step 150 | Loss 0.2657989263534546
Step 160 | Loss 0.2594328820705414
Step 170 | Loss 0.24827778339385986
Step 180 | Loss 0.2545925974845886
Step 190 | Loss 0.2500864267349243
Step 200 | Loss 0.2514670193195343
Step 210 | Loss 0.2509611248970032
Step 220 | Loss 0.2456578016281128
Step 230 | Loss 0.24389488995075226
Step 240 | Loss 0.2426615059375763
Step 250 | Loss 0.24042271077632904
Step 260 | Loss 0.2375590205192566
Step 270 | Loss 0.23877790570259094
Step 280 | Loss 0.23

In [6]:
model_aire_no.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  102 neurons
Hidden layer  2:  102 neurons
Hidden layer  3:  102 neurons
Hidden layer  4:  102 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_AIRe.obj", model=model_aire_no, resolution=128, level=0.0)

## DepGraph: Model training (no densification)

In [4]:
set_seeds()
model_dep_no = si.SIRENSDF()
prune_DepGraph = pm.DepGraph(model_dep_no, 0.6, int(size_per_layer * 0.2))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_dep_no, pruning_module=prune_DepGraph)
optimizer = torch.optim.Adam(model_dep_no.parameters(), lr=1e-4)

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_dep_no, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_DepGraph)

Step 0 | Loss 2.5599801540374756
Step 10 | Loss 0.47809037566185
Step 20 | Loss 0.3789999783039093
Step 30 | Loss 0.33777710795402527
Step 40 | Loss 0.318610280752182
Step 50 | Loss 0.30183839797973633
Step 60 | Loss 0.2956336438655853
Step 70 | Loss 0.28242257237434387
Step 80 | Loss 0.27570125460624695
Step 90 | Loss 0.2705899775028229
Step 100 | Loss 0.2663497030735016
Step 110 | Loss 0.26338595151901245
Step 120 | Loss 0.25615599751472473
Step 130 | Loss 0.25628557801246643
Step 140 | Loss 0.2515798807144165
Step 150 | Loss 0.25408610701560974
Step 160 | Loss 0.24776270985603333
Step 170 | Loss 0.23666636645793915
Step 180 | Loss 0.24300862848758698
Step 190 | Loss 0.23860269784927368
Step 200 | Loss 0.2400287687778473
Step 210 | Loss 0.23971779644489288
Step 220 | Loss 0.23428228497505188
Step 230 | Loss 0.23260390758514404
Step 240 | Loss 0.23160606622695923
Step 250 | Loss 0.22927217185497284
Step 260 | Loss 0.2264605164527893
Step 270 | Loss 0.22785885632038116
Step 280 | Loss 

In [6]:
model_dep_no.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  256 neurons
Hidden layer  1:  102 neurons
Hidden layer  2:  102 neurons
Hidden layer  3:  102 neurons
Hidden layer  4:  102 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_DepGraph.obj", model=model_dep_no, resolution=128, level=0.0)

## AIRe: Model training with densification

In [6]:
set_seeds()
model_aire_yes = si.SIRENSDF()
prune_AIRe = pm.AIRe(model_aire_yes, 0.6, int(size_per_layer * 0.2))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_aire_yes, pruning_module=prune_AIRe)
optimizer = torch.optim.Adam(model_aire_yes.parameters(), lr=1e-4)

In [7]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_aire_yes, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_AIRe, densification=True)

Step 0 | Loss 2.571408987045288
Step 10 | Loss 0.48973679542541504
Step 20 | Loss 0.39074957370758057
Step 30 | Loss 0.34962552785873413
Step 40 | Loss 0.330488920211792
Step 50 | Loss 0.31372010707855225
Step 60 | Loss 0.3075138330459595
Step 70 | Loss 0.2943316400051117
Step 80 | Loss 0.28757527470588684
Step 90 | Loss 0.28246963024139404
Step 100 | Loss 0.27817487716674805
Step 110 | Loss 0.2752159535884857
Step 120 | Loss 0.26794353127479553
Step 130 | Loss 0.2681148052215576
Step 140 | Loss 0.26337018609046936
Step 150 | Loss 0.2657989263534546
Step 160 | Loss 0.2594328820705414
Step 170 | Loss 0.24827778339385986
Step 180 | Loss 0.2545925974845886
Step 190 | Loss 0.2500864267349243
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.2514670193195343
Step 210 | Loss 0.3059181571006775
Step 220 | Loss 0.2800002992153168
Step 230 | Loss 0.26505109667778015
Step 240 | Loss 0.2552188038825989
Step 250 | Loss 0.24738073348999023
Step 260 | Loss 0.2404773235321045
Step 270 | 

In [8]:
model_aire_yes.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  102 neurons
Hidden layer  2:  102 neurons
Hidden layer  3:  102 neurons
Hidden layer  4:  102 neurons
Final layer    :    1 neurons


In [9]:
marching_cubes.write_obj("armadillo_128_AIRe_densified.obj", model=model_aire_yes, resolution=128, level=0.0)

## DepGraph: Model training with densification

In [4]:
set_seeds()
model_dep_yes = si.SIRENSDF()
prune_DepGraph = pm.DepGraph(model_dep_yes, 0.6, int(0.2*size_per_layer))
model_loss = loss.Loss(lambda_surface=175,  lambda_eikonal=0.25, lambda_normal=2, lambda_inter=0.05, lambda_off=2, lambda_twd=1e-4, model=model_dep_yes, pruning_module=prune_DepGraph)
optimizer = torch.optim.Adam(model_dep_yes.parameters(), lr=1e-4)

In [5]:
train(epochs=1000, data=mesh, no_surface=10000, no_off_surface=15000, model=model_dep_yes, loss=model_loss, optimizer=optimizer, scene=mesh.scene, pruning_module=prune_DepGraph, densification=True)

Step 0 | Loss 2.5599801540374756
Step 10 | Loss 0.47809037566185
Step 20 | Loss 0.3789999783039093
Step 30 | Loss 0.33777710795402527
Step 40 | Loss 0.318610280752182
Step 50 | Loss 0.30183839797973633
Step 60 | Loss 0.2956336438655853
Step 70 | Loss 0.28242257237434387
Step 80 | Loss 0.27570125460624695
Step 90 | Loss 0.2705899775028229
Step 100 | Loss 0.2663497030735016
Step 110 | Loss 0.26338595151901245
Step 120 | Loss 0.25615599751472473
Step 130 | Loss 0.25628557801246643
Step 140 | Loss 0.2515798807144165
Step 150 | Loss 0.25408610701560974
Step 160 | Loss 0.24776270985603333
Step 170 | Loss 0.23666636645793915
Step 180 | Loss 0.24300862848758698
Step 190 | Loss 0.23860269784927368
Added 52 frequencies to the embedding layer.
Step 200 | Loss 0.2400287687778473
Step 210 | Loss 0.29678577184677124
Step 220 | Loss 0.2712213397026062
Step 230 | Loss 0.25614631175994873
Step 240 | Loss 0.24642033874988556
Step 250 | Loss 0.23889096081256866
Step 260 | Loss 0.23209798336029053
Step 27

In [6]:
model_dep_yes.neuron_counts()

Neuron counts per layer:
----------------------------------------
Hidden layer  0:  308 neurons
Hidden layer  1:  102 neurons
Hidden layer  2:  102 neurons
Hidden layer  3:  102 neurons
Hidden layer  4:  102 neurons
Final layer    :    1 neurons


In [7]:
marching_cubes.write_obj("armadillo_128_DepGraph_densified.obj", model=model_dep_yes, resolution=128, level=0.0)

In [24]:
#from src.mesh_extraction import marching_cubes_test 
#marching_cubes_test.write_obj("lucy_128_gt.obj", mesh.scene, 128, 0.0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Sample a few surface points and check their SDF values
test_points = mesh.vertices[:10]  # First 10 points
test_tensor = torch.from_numpy(test_points).float().to(device)
with torch.no_grad():
    sdf_values = model_no(test_tensor)
print("SDF values for surface points:")
print(sdf_values)
print("Mean absolute SDF:", torch.abs(sdf_values).mean().item())

SDF values for surface points:
tensor([[ 2.1020e-05],
        [ 3.4788e-03],
        [-2.3386e-04],
        [ 3.9918e-05],
        [ 4.1725e-03],
        [ 4.5397e-03],
        [ 4.0368e-03],
        [-4.1481e-06],
        [ 1.9721e-03],
        [-3.3731e-03]], device='cuda:0')
Mean absolute SDF: 0.002187199890613556


In [ ]:
# import src.mesh_extraction.marching_cubes_test as marching_cubes_test
# marching_cubes_test.write_obj("armadillo_mesh_ground_truth_128.obj", scene=mesh.scene, resolution=128, level=0.0)


In [5]:
# Make batched point
device = torch.device("cuda")
test_point = torch.from_numpy(np.array([[-1, -1, -1], [0, 0, 0]])).float().to(device)

# Compute SIREN model prediction
sdf_pred = model_no(test_point)
print("SIREN prediction:", sdf_pred)

# Compute Open3D signed distance
distance = mesh.scene.compute_signed_distance(
    o3d.core.Tensor(test_point.cpu().numpy(), dtype=o3d.core.Dtype.Float32)
)
print("Open3D SDF:", distance.numpy())

SIREN prediction: tensor([[ 0.0181],
        [-0.0433]], device='cuda:0', grad_fn=<AddmmBackward0>)
Open3D SDF: [ 0.96991843 -0.07447068]


In [25]:
torch.save(model_no.state_dict(), "model_no.pth")
torch.save(model_yes.state_dict(), "model_yes.pth")
torch.save(model_aire_no.state_dict(), "model_aire_no.pth")
torch.save(model_dep_no.state_dict(), "model_dep_no.pth")
torch.save(model_aire_yes.state_dict(), "model_aire_yes.pth")
torch.save(model_dep_yes.state_dict(), "model_dep_yes.pth")

In [11]:
model = si.SIRENSDF()
model.load_state_dict(torch.load('model_no.pth', weights_only=True))
model.eval()
model.to(device=torch.device("cuda"))


marching_cubes.write_obj("armadillo_256_unpruned.obj", model=model, resolution=256)
